# Recommender Metrics

Review recommender model metrics and scores.

Steps:
- Load recommender metrics files.
- Inspect ratings and score distributions.
- Summarize evaluation outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

summary = {
    'metrics': {},
}

metrics_files = [
    REPO_ROOT / 'experiments' / 'recommender' / 'metrics' / 'metrics.json',
    REPO_ROOT / 'experiments' / 'recommender' / 'metrics' / 'recommender_metrics.json',
]

for path in metrics_files:
    if not path.exists():
        print('Missing:', path)
        continue
    data = json.loads(path.read_text(encoding='utf-8'))
    summary['metrics'][path.name] = data
    print(path.name, data)


In [ ]:
# Inspect any recommender score outputs.
metrics_csv = REPO_ROOT / 'experiments' / 'recommender' / 'metrics' / 'metrics.csv'
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    summary['metrics']['metrics_csv'] = {
        'rows': int(df.shape[0]),
        'cols': int(df.shape[1]),
        'columns': list(df.columns),
    }
    print('Metrics CSV:', df.shape)
    print(df.head(10))
else:
    print('Missing metrics.csv')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_recommender_metrics_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
